# SVD Embeddings

AI & ML Foundations: Embeddings and retrieval · *Mathematics for AI and Machine Learning* by Fuheng Wu

Embed the rows of a rating matrix as U_k Sigma_k: similar items land together and inner products are kept (Chapter 8).

Companion code for the [interactive demo](https://math4ai.org/companion/ml/svd-embedding/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(2)

# The demo's setup: 12 movies (6 action, 6 drama) x 16 users, built from two hidden tastes
movies = ["Speed Run", "Iron Fist", "Night Raid", "Blast Zone", "Edge Line", "Hard Drive",
          "Quiet Rooms", "Letters", "The Garden", "Long Winter", "Old Friends", "Small Town"]
genre = torch.tensor([0] * 6 + [1] * 6)
k, noise = 2, 0.3
F_ = torch.where(genre[:, None] == 0, torch.tensor([1.0, 0.15]), torch.tensor([0.15, 1.0])) + 0.1 * torch.randn(12, 2)
P = torch.randn(16, 2).abs() + 0.2
A = 2 * F_ @ P.T + noise * torch.randn(12, 16)        # ratings: rows are movies

# ── SVD embedding of the rows: A V_k = U_k Sigma_k (Chapter 8) ───────────────
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
E = U[:, :k] * S[:k]
print("singular values:", [round(s, 2) for s in S[:5].tolist()], "...")
print("A V_k == U_k Sigma_k:", torch.allclose(A @ Vh[:k].T, E, atol=1e-5))

# Full coordinates keep every inner product: (AV)(AV)^T = A A^T
full = A @ Vh.T
print("(AV)(AV)^T == A A^T:", torch.allclose(full @ full.T, A @ A.T, atol=1e-3))

# ── Truncation: error in the inner products vs k (Eckart-Young) ────────────
G = A @ A.T
for kk in range(1, 7):
    Ek = U[:, :kk] * S[:kk]
    err = (torch.linalg.matrix_norm(G - Ek @ Ek.T) / torch.linalg.matrix_norm(G)).item()
    print(f"k = {kk}: relative inner-product error {err:.4f}")

# ── Nearest neighbours in the 2D embedding stay within the genre ────────────
En = torch.nn.functional.normalize(E, dim=1)
sim = En @ En.T
sim.fill_diagonal_(-2)
nn_idx = sim.argmax(1)
print(f"\nwith k = {k}: {(genre[nn_idx] == genre).sum().item()} of 12 movies have a same-genre nearest neighbour")
for i in (0, 6):
    print(f"  {movies[i]:12s} -> {movies[nn_idx[i]]}")

singular values: [37.39, 8.71, 1.79, 1.37, 1.32] ...
A V_k == U_k Sigma_k: True
(AV)(AV)^T == A A^T: True
k = 1: relative inner-product error 0.0543
k = 2: relative inner-product error 0.0033
k = 3: relative inner-product error 0.0024
k = 4: relative inner-product error 0.0019
k = 5: relative inner-product error 0.0015
k = 6: relative inner-product error 0.0011

with k = 2: 12 of 12 movies have a same-genre nearest neighbour
  Speed Run    -> Edge Line
  Quiet Rooms  -> The Garden
